# Tutorial for parameter inference with SBI

We use here a simulation-based inference (sbi) framework (see https://sbi-dev.github.io/sbi/ ), a supervised learning approach where a network is trained to learn the mapping between simulated data and the posterior distribution of the model parameters used to simulate them.

The `pypopsyn/learning/train_sbi.py` script employs a method called Neural Posterior Estimation (NPE) that allows to train a neural density estimator over a dataset of samples of simulated neutron star populations to directly approximate the posterior distributions of the input parameters.
In this case the inference is amortized, meaning that the trained model is able to predict a posterior distribution for any input simulated population of neutron stars.

To create the dataset of simulations for this training experiment you can use the notebooks `04_simulation_helper_tutorial.ipynb` and `05_generator_tutorial.ipynb` in `tutorials/tutorial_notebooks` or alternatively you can use the dataset that is already stored in the directory path `data/example_generator_magrot`.

Once the dataset containing the heatmaps or 2D arrays of various simulated populations has been created, to train the network over the dataset one can run the script:
```
python pypopsyn/learning/train_sbi.py --configuration config_train_sbi.json
```

where the `config_train_sbi.json` file contains all the information needed by the network to train. You can customize the `config_train_sbi.json` according to your needs.

You can look at the paper [Graber et al. 2024](https://ui.adsabs.harvard.edu/abs/2024ApJ...968...16G/abstract) for an application of this method to infer the properties of the observed radio pulsar population.

In [ ]:
import collections
import corner
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
import torch

from matplotlib.ticker import ScalarFormatter

import utilities.plot_settings

## Run the training script

In [ ]:
!python ../../pypopsyn/learning/train_sbi.py --configuration config_train_sbi.json

## Extract training results

In the following we extract the training and validation losses in order to see how they evolve over the training process. 
NOTE: Make sure to change the name of the folders `train_log_output_path` and `model_output_path` where your training results are saved.

In [ ]:
with open(f"config_train_sbi.json", "r") as read_file:
    config_train = json.load(read_file)

train_output_path = config_train["trainer"]["save_dir"]
train_log_output_path = f"{train_output_path}/logs/SBI_ConvolutionMDN/20240809_15165/"
model_output_path = f"{train_output_path}/models/SBI_ConvolutionMDN/20240809_151656/"

In [ ]:
with open(f"{train_log_output_path}training_statistics.json", "r") as read_file:
    learning_data = json.load(read_file)

Plot the training and validation losses as a function of the training epoch for the two parameters to predict.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel(r"Epoch")
ax.set_ylabel(r"Accuracy")
ax.plot(
    learning_data["training_log_probs"]["step"],
    learning_data["training_log_probs"]["value"],
    linestyle="-",
    linewidth=4,
    color="tab:blue",
    rasterized=True,
    label="training",
)
ax.plot(
    learning_data["validation_log_probs"]["step"],
    learning_data["validation_log_probs"]["value"],
    linestyle="-",
    linewidth=4,
    color="tab:orange",
    rasterized=True,
    label="validation",
)
plt.legend(
    bbox_to_anchor=(1.05, 1), frameon=False, loc=0
)

## Perform inference with the trained model

Once the model has been trained we can use it to perform inference on any new input sample.
Here we show the performance of the model in predicting the four parameters on the test dataset.

In [ ]:
# define the path to the trained model.
trained_model_path = f"{model_output_path}trained_model.pickle"

Run the inference script.

In [ ]:
# Construct the command.
command = f"python ../../pypopsyn/learning/infer_sbi.py --configuration config_train_sbi.json --trained_model {trained_model_path} --corner_plot True"

# Execute the command.
!{command}

## Extract inference results

In the following we extract the inference results on the test dataset. 
NOTE: Make sure to change the name of the folder `inference_log_output_path` where your inference results are saved.

In [ ]:
inference_output_path = config_train["infer"]["save_dir"]
inference_log_output_path = f"{inference_output_path}/logs/SBI_ConvolutionMDN/20240729_161756/"
test_dataset_path = config_train["test_data_loader"]["dataset_path"]

Load the saved posterior samples for one of the test samples to plot the corner plot.

In [ ]:
test_dataset = pd.read_csv(test_dataset_path)

In [ ]:
# Choose a specific test sample, i.e., choose an index between 0 and 3 for 4 test samples.
sample_index = 0

# Extract the ground truth values of the parameters and the posterior samples.
ground_truth = test_dataset.iloc[sample_index][["B_initial_log10_mean","P_initial_log10_mean"]].values
posterior_samples = torch.load(f"{inference_log_output_path}samples_{sample_index}.pt")

print("ground_truth B_initial_log10_mean: ", ground_truth[0])
print("ground_truth P_initial_log10_mean: ", ground_truth[1])

In [ ]:
# Calculate the quantiles of the posterior samples to extract the median values for the parameters and the 95% credibility interval.
quantile = np.quantile(posterior_samples, [0.025, 0.5, 0.975], axis=0)

param_median = quantile[1]
param_inf = quantile[0]
param_sup = quantile[2]

param_err_inf = param_median - param_inf
param_err_sup = param_sup - param_median

print("predicted B_initial_log10_mean: ", f"{param_median[0]} + {param_err_sup[0]} - {param_err_inf[0]}")
print("predicted P_initial_log10_mean: ", f"{param_median[1]} + {param_err_sup[1]} - {param_err_inf[1]}")

## Plot the corner plot

In [ ]:
parameter_ranges = [[12, 14], [-1.5, -0.3]]
parameter_labels = [r"$\mu_{\log B}$", r"$\mu_{\log P}$"]

fig = plt.figure(figsize=(10, 10))

figure = corner.corner(
    posterior_samples.numpy(),
    bins=32,
    labels=parameter_labels,
    #label_kwargs={"fontsize": 36},
    range=parameter_ranges,
    quantiles=[0.025, 0.5, 0.975],
    levels=(
        1 - np.exp(-0.5),
        1 - np.exp(-2),
        1 - np.exp(-9.0 / 2.0),
    ),  # 1, 2 and 3 sigma levels
    show_titles=True,
    #title_kwargs={"fontsize": 16},
    fig = fig
)
corner.overplot_lines(
    figure, param_median, color="tab:red"
)
corner.overplot_lines(
    figure, ground_truth, color="tab:blue"
)
corner.overplot_points(
    figure,
    param_median[None],
    marker="s",
    color="tab:red",
)
corner.overplot_points(
    figure,
    ground_truth[None],
    marker="s",
    color="tab:blue",
)

## Coverage probability test

A test that can be performed to check if the trained density estimator is able to produce well calibrated posteriors is the coverage probability test. Let's consider a dataset $\{ (\boldsymbol{\theta}_i, \boldsymbol{x}_i) \}$ of simulations $\boldsymbol{x}$ that are properly labeled by the respective simulation parameters $\boldsymbol{\theta}$. We compute the posterior $\mathcal{P}(\boldsymbol{\theta}|\boldsymbol{x}_i)$ for each $\boldsymbol{x}_i$. For each posterior, we then define a credible region $\Theta_i$ with the smallest volume in the multidimensional parameter space of $\boldsymbol{\theta}$ corresponding to a total probability $1 - \alpha$ with $\alpha \in [0, 1]$:
$$
\int_{\Theta_i} \mathcal{P}(\boldsymbol{\theta}|\boldsymbol{x}_i) d \boldsymbol{\theta} = 1 - \alpha, 
$$
where $1 - \alpha$ defines the so-called credibility level.
Considering the region with the smallest volume guarantees that we are focusing on the region of the parameter space that encloses the parameter values $\boldsymbol{\theta}$ with the highest posterior probability density. In the literature, this is also called the highest posterior density region.

By counting how many $\boldsymbol{\theta}_i$ fall inside the corresponding credibility regions $\Theta_i$ we obtain a measure of how well the estimated posteriors are able to recover the label parameters $\boldsymbol{\theta}$. This count gives an estimate of the so-called coverage probability. If the posterior estimator is well calibrated, the coverage probability should be equal to the value $1 - \alpha$. If the coverage probability is higher than $1 - \alpha$, this is a symptom for a posterior estimator that tends to generate posteriors that are too conservative. On the other hand, if the coverage probability is lower than $1 - \alpha$, this indicates that the estimated posteriors are overconfident.

Load the coverage probability results. Note that in this small example we have only 4 test simulations, a larger number of simulation samples would be required to properly perform a coverage test. This is shown only for illustrative purposes.

In [ ]:
coverage_probability = np.load(f"{inference_log_output_path}coverage_probability.npy")
betas = np.linspace(0, 1, len(coverage_probability))

In [ ]:
# Plot the coverage.
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(
    betas,
    coverage_probability,
    color="steelblue",
    linewidth=3,
    label="upper right",
)
ax.plot([0, 1], [0, 1], color="k", linestyle="--")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel(r"Credibility level $1-\alpha$")
ax.set_ylabel(r"Coverage probability")

Once the model is trained and is able to produce well calibrated posteriors it can be applied on the real observed population.
To do so one needs only to specify the path to the generated maps from the ATNF catalgue in the configuration file `config_train_sbi.json` in the field related to the `test_data_loader`. Make sure also to specify the correct labels that you want to predict through the `filter_labels` option.